In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

FRAME_SIZE = 40960       # The input size for U-Net
SAMPLES_PER_FILE = 50    # Expansion Factor (70 files * 50 = 3500 samples)
AUGMENTATION_RATIO = 0.3 # 30% Augmented, 70% Identity

np.random.seed(42)

INPUT_FILE = UNET_TRAIN_DIR / f"{CommSignal2}_train.npy"
OUTPUT_FILE = UNET_TRAIN_DIR / f"{CommSignal2}_train_augmented.npy"
METADATA_FILE = UNET_TRAIN_DIR / f"{CommSignal2}_metadata.csv"

In [ ]:
def apply_phase_rotation(signal):
    """
    Applies random phase shift: x_new = x * e^(j * theta)
    Rotates constellation without changing geometry.
    """
    theta = np.random.uniform(0, 2 * np.pi)
    rotation_complex = np.exp(1j * theta)
    
    # Signal is (2, N). Convert to complex -> rotate -> back to (2, N)
    complex_sig = signal[0] + 1j * signal[1]
    rotated_sig = complex_sig * rotation_complex
    
    return np.stack([rotated_sig.real, rotated_sig.imag]), theta

def apply_conjugate_flip(signal):
    """
    Applies spectral inversion: x_new = conjugate(x)
    """
    flipped = signal.copy()
    flipped[1] *= -1 # Invert Q channel
    return flipped

def get_random_slice(full_signal, frame_size):
    """
    Extracts a random valid slice from the source signal.
    """
    total_samples = full_signal.shape[1]
    max_start = total_samples - frame_size
    
    if max_start < 0:
        raise ValueError(f"Signal too short! Needed {frame_size}, got {total_samples}")
        
    start_idx = np.random.randint(0, max_start + 1)
    return full_signal[:, start_idx : start_idx + frame_size], start_idx

In [ ]:
print(f"Loading Source: {INPUT_FILE}...")
raw_data = np.load(INPUT_FILE)

# Ensure Shape is (N_Files, 2, N_Samples)
if raw_data.shape[1] != 2:
    print(f"Transposing input shape from {raw_data.shape}...")
    raw_data = raw_data.transpose(0, 2, 1)

num_files = raw_data.shape[0]
print(f"Loaded {num_files} source recordings.")

metadata_rows = []
all_samples = []

for file_idx in tqdm(range(num_files), desc="Augmenting"):
    source_signal = raw_data[file_idx]
    
    for sample_i in range(SAMPLES_PER_FILE):
        # Slice
        slice_data, start_idx = get_random_slice(source_signal, FRAME_SIZE)
        
        # Decision Fork
        is_augmented = False
        aug_type = "Identity"
        theta = 0.0
        is_flipped = False
        
        # 30% Chance to Apply Physics
        if np.random.rand() < AUGMENTATION_RATIO:
            is_augmented = True
            
            # Rotation (Always)
            slice_data, theta = apply_phase_rotation(slice_data)
            aug_type = "Rotated"
            
            # Flip (50% conditional)
            if np.random.rand() < 0.5:
                slice_data = apply_conjugate_flip(slice_data)
                is_flipped = True
                aug_type = "Rotated+Flipped"

        
        all_samples.append(slice_data.astype(np.float32))

        metadata_rows.append({
            'sample_index': len(all_samples) - 1,
            'parent_id': file_idx,
            'start_idx': start_idx,
            'is_augmented': is_augmented,
            'aug_type': aug_type,
            'theta': theta,
            'flipped': is_flipped
        })

print("Stacking into single tensor...")
final_dataset = np.stack(all_samples)

print(f"Saving to {OUTPUT_FILE}...")
np.save(OUTPUT_FILE, final_dataset)

df = pd.DataFrame(metadata_rows)
df.to_csv(METADATA_FILE, index=False)
print(f"Metadata saved to {METADATA_FILE}")

print(f"Final Tensor Shape: {final_dataset.shape}")
print(f"Metadata Rows: {len(df)}")
print(f"Augmented Ratio: {df['is_augmented'].mean():.2%}")